## 
## **--Project: Superstore Retail Sales EDA--**
## 

#### **1. IMPORT LIBRARIES AND LOAD DATASET** 

##### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


##### Load Dataset

In [ ]:
# Read the CSV file and store it in a DataFrame
df = pd.read_csv("superstore_Retail_sales.csv")

#### **2. DATA UNDERSTANDING**

In [ ]:
#Display the first five rows
df.head()

In [ ]:
#Check the number of rows and columns
print("\nDataset Shape:", df.shape,"\n")

#Check dataset information
df.info()

#Check data types of all columns
print("\nData Types:")
print(df.dtypes)

#Generate statistical summary
print("\n\nStatistical Summary:")
df.describe()

#### **3. HANDLE MISSING VALUES**

##### Cheak Missing Value

In [ ]:
# Count missing values in each column
print(df.isnull().sum())

# Count all missing values in the dataset
print("Total missing values:", df.isnull().sum().sum())

##### Handle Missing Values

In [ ]:
# Check rows where all important fields are missing
key_columns = ["Order ID", "Order Date", "Sales", "Profit"]

print("Rows with missing key fields:")
print(df[key_columns].isnull().sum())

# Remove rows where all key fields are missing
df = df.dropna(subset=key_columns, how="all").copy()

# Convert Order Date into datetime format
df["Order Date"] = pd.to_datetime(
    df["Order Date"], errors="coerce"
)

# Fill missing categorical values
# only where appropriate
for col in ["Category", "Region", "Segment"]:
    df[col] = df[col].fillna("Unknown")

# Check remaining missing values
print("Dataset shape:", df.shape)
print(df.isnull().sum())

In [ ]:
# Check records with missing Sales, Profit, or Order Date
print(
    df[["Order Date", "Sales", "Profit"]]
    .isnull()
    .sum()
)

# Inspect records with missing Sales
df[df["Sales"].isna()].head()

In [ ]:
# Remove records that cannot be used for sales analysis
df = df.dropna(
    subset=["Order Date", "Sales", "Profit"]
).copy()

# Verify the cleaned dataset
print("Cleaned dataset shape:", df.shape)
print("Missing values:")
print(df.isnull().sum())

In [ ]:
# Count all missing values in the dataset
print("Total missing values:", df.isnull().sum().sum())

**Note on Postal Code Missing Values**

After cleaning the dataset, 11 missing values remain in the Postal Code column. These values have not been filled because Postal Code is not required for the current analysis of Sales, Profit, Category, Region, and Customer Segment. Filling these values without reliable information could introduce incorrect data.

#### **4. PARSE ORDER DATES & EXTRACT YEAR-MONTH**

##### Convert date into Year-Month

In [ ]:
# Convert Order Date into datetime format
df["Order Date"] = pd.to_datetime(
    df["Order Date"],
    errors="coerce"
)

# Extract the year and month
df["Year-Month"] = (
    df["Order Date"].dt.to_period("M")
)

# Display the updated columns
df[["Order Date", "Year-Month"]].head()

#### **5. GRPUP BY ANALYSIS**

In [ ]:
# Display all column names
print(df.columns.tolist())

##### Category-wise Sales and Profit

In [ ]:
# Group the data by product category
category_analysis = df.groupby("Category").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum")
).reset_index()

# Sort by total sales in descending order
category_analysis = category_analysis.sort_values(
    by="Total_Sales",
    ascending=False
)

# Display the results
category_analysis

##### Region-wise Sales and Profit

In [ ]:
# Group the data by region
region_analysis = df.groupby("Region").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum")
).reset_index()

# Sort by total sales
region_analysis = region_analysis.sort_values(
    by="Total_Sales",
    ascending=False
)

# Display the results
region_analysis

##### Customer Segment-wise Sales and Profit

In [ ]:
# Group the data by customer segment
segment_analysis = df.groupby("Segment").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum")
).reset_index()

# Sort by total sales
segment_analysis = segment_analysis.sort_values(
    by="Total_Sales",
    ascending=False
)

# Display the results
segment_analysis

##### Combined Analysis

In [ ]:
# Group by category, region, and customer segment
combined_analysis = df.groupby(
    ["Category", "Region", "Segment"]
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum")
).reset_index()

# Display the first 10 combinations
combined_analysis.head(10)

#### **6. DATA VISUALIZATION**

##### Plot Monthly Sales Trends

In [ ]:
# Exclude rows with missing order dates
monthly_data = df.dropna(
    subset=["Order Date"]
).copy()

# Extract year and month
monthly_data["Year-Month"] = (
    monthly_data["Order Date"].dt.to_period("M")
)

# Calculate total sales for each month
monthly_sales = monthly_data.groupby(
    "Year-Month"
)["Sales"].sum().reset_index()

# Convert the month values to strings for plotting
monthly_sales["Year-Month"] = (
    monthly_sales["Year-Month"].astype(str)
)

# Create the monthly sales line chart
plt.figure(figsize=(14, 6))

sns.lineplot(
    data=monthly_sales,
    x="Year-Month",
    y="Sales",
    marker="o"
)

plt.title("Monthly Sales Trend")
plt.xlabel("Year-Month")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

##### Find the highest sales in Month

In [ ]:
# Find the month with the highest sales
monthly_data = df.dropna(subset=["Order Date"]).copy()

monthly_sales = (
    monthly_data.groupby(
        monthly_data["Order Date"].dt.to_period("M")
    )["Sales"]
    .sum()
)

print("Highest sales month:", monthly_sales.idxmax())
print("Highest monthly sales:", round(monthly_sales.max(), 2))

# Check the relationship between discount and profit
print(
    "Discount-Profit correlation:",
    round(df["Discount"].corr(df["Profit"]), 3)
)

##### Plot Discount vs. Profit

In [ ]:
# Remove rows with missing discount or profit values
discount_data = df.dropna(
    subset=["Discount", "Profit"]
).copy()

# Create the scatter plot
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=discount_data,
    x="Discount",
    y="Profit",
    alpha=0.5
)

# Add a zero-profit reference line
plt.axhline(
    y=0,
    color="red",
    linestyle="--",
    label="Zero Profit"
)

plt.title("Discount vs. Profit Analysis")
plt.xlabel("Discount")
plt.ylabel("Profit")
plt.legend()
plt.tight_layout()
plt.show()

##### Category-wise Sales Percentage

In [ ]:
# Calculate total sales for each product category
category_sales = df.groupby("Category")["Sales"].sum()

# Calculate the percentage contribution of each category
sales_percentage = (
    category_sales / category_sales.sum()
) * 100

# Create a pie chart
plt.figure(figsize=(8, 8))

plt.pie(
    sales_percentage,
    labels=sales_percentage.index,
    autopct="%1.1f%%",
    startangle=90
)

# Add chart title
plt.title("Product Category-wise Sales Percentage")

# Display the pie chart
plt.show()

##### Year-wise Sales

In [ ]:
# Extract the year from Order Date
df["Year"] = df["Order Date"].dt.year

# Calculate total sales for each year and category
year_category_sales = df.groupby(
    ["Year", "Category"]
)["Sales"].sum().reset_index()

# Create a grouped bar chart
plt.figure(figsize=(12, 6))

ax = sns.barplot(
    data=year_category_sales,
    x="Year",
    y="Sales",
    hue="Category"
)

# Display sales values above each bar
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.0f",
        padding=3,
        fontsize=10 
    )

# Add chart title and axis labels
plt.title("Year-wise Sales by Product Category")
plt.xlabel("Year")
plt.ylabel("Total Sales")

# Display the legend
plt.legend(title="Product Category")

# Add space above the bars for the labels
plt.tight_layout()

# Display the chart
plt.show()

#### **7. BUSINESS OBSEVATIONS**

**1. Category-wise Sales and Profit**

Technology has the highest sales of about $836,154 and the highest profit of about $145,455. Furniture has sales of about $742,000, but its profit is only about $18,451. This means that more sales do not always mean more profit.

**2. Region-wise Performance**

The West region has the highest sales of about $725,458 and the highest profit of about $108,418. The South region has the lowest sales of about $391,722. The business can learn from the West region to improve sales in other regions.

**3. Customer Segment Analysis**

The Consumer segment has the highest sales of about $1.16 million and the highest profit of about $134,119. Corporate customers have sales of about $706,146, while Home Office customers have sales of about $429,653. This shows that Consumer customers are the biggest contributors to sales.

**4. Monthly Sales Trend**

November 2018 had the highest monthly sales of $118,447.82. This shows that sales were highest in this month. The business can use this information to plan stock and marketing activities for busy months.

**5. Discount and Profit Relationship**

The correlation between Discount and Profit is about -0.219. This shows a weak negative relationship between them. In this dataset, higher discounts are generally linked to lower profits. However, this does not mean discounts are the only reason for lower profit.

**Conclusion**

Technology and the West region perform well in sales and profit. Furniture has lower profit compared to its sales. The business should review its discount plans and look for ways to increase profit.